# EchoNext Single-Lead Colab Starter

This notebook is designed for a Google-Drive-first workflow:
- code lives in Colab under `/content/echonext_single_lead`
- the EchoNext dataset lives in Google Drive
- train/val/test files are copied into local runtime storage for faster model training


## 1. Runtime setup

In Colab, set `Runtime -> Change runtime type -> GPU` if available.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/echonext_single_lead')
LOCAL_DATA_DIR = PROJECT_DIR / 'data' / 'raw'
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/echonext-a-dataset-for-detecting-echocardiogram-confirmed-structural-heart-disease-from-ecgs-1.1.1')
LABEL = 'shd_moderate_or_greater_flag'
TRAIN_FRACTION = 0.01
SEED = 42

print('PROJECT_DIR =', PROJECT_DIR)
print('LOCAL_DATA_DIR =', LOCAL_DATA_DIR)
print('DRIVE_DATA_DIR =', DRIVE_DATA_DIR)
print('LABEL =', LABEL)
print('TRAIN_FRACTION =', TRAIN_FRACTION)


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Get the project code into Colab

Recommended: keep the code on GitHub and clone it here.

If you prefer, you can instead copy the project folder from Drive.

In [ ]:
# Option A: clone from GitHub
# !git clone https://github.com/your-org/your-repo.git /content/echonext_single_lead

# Default option for this workflow: copy project folder from Google Drive
!rm -rf /content/echonext_single_lead
!cp -r /content/drive/MyDrive/echonext_single_lead /content/echonext_single_lead

print('Project copied from Google Drive to /content/echonext_single_lead')


## 4. Install dependencies

In [ ]:
assert PROJECT_DIR.exists(), f'Project directory not found: {PROJECT_DIR}'
!pip install -r /content/echonext_single_lead/requirements.txt


In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 5. Verify the dataset in Google Drive

For the first project version, we only need the train/val/test files plus the metadata CSV. We do not need the `no_split` files in Colab.

In [ ]:
expected_drive_files = [
    'echonext_metadata_100k.csv',
    'EchoNext_train_waveforms.npy',
    'EchoNext_val_waveforms.npy',
    'EchoNext_test_waveforms.npy',
    'EchoNext_train_tabular_features.npy',
    'EchoNext_val_tabular_features.npy',
    'EchoNext_test_tabular_features.npy',
]

print('Drive data directory exists:', DRIVE_DATA_DIR.exists())
if DRIVE_DATA_DIR.exists():
    print('Files present in Drive directory:')
    for path in sorted(DRIVE_DATA_DIR.glob('*')):
        print(' -', path.name)

missing = [name for name in expected_drive_files if not (DRIVE_DATA_DIR / name).exists()]
if missing:
    print('\nMissing required files in Drive:')
    for name in missing:
        print(' -', name)
else:
    print('\nAll required Drive files found.')


## 6. Copy required files from Drive into local runtime storage

This is recommended because training from `/content` is usually faster and more reliable than reading large arrays directly from Google Drive.

In [ ]:
import shutil

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
required_files = [
    'echonext_metadata_100k.csv',
    'EchoNext_train_waveforms.npy',
    'EchoNext_val_waveforms.npy',
    'EchoNext_test_waveforms.npy',
    'EchoNext_train_tabular_features.npy',
    'EchoNext_val_tabular_features.npy',
    'EchoNext_test_tabular_features.npy',
]

for name in required_files:
    src = DRIVE_DATA_DIR / name
    dst = LOCAL_DATA_DIR / name
    if not src.exists():
        raise FileNotFoundError(f'Missing source file: {src}')
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print('Skipping existing file:', name)
        continue
    print('Copying', name)
    shutil.copy2(src, dst)

print('\nLocal data directory contents:')
for path in sorted(LOCAL_DATA_DIR.glob('*')):
    print(' -', path.name, path.stat().st_size)


## 7. Inspect the local copied dataset

In [ ]:
%cd /content/echonext_single_lead
!python3 scripts/01_inspect_data.py --data_dir data/raw


## 8. Quick smoke test on a small fraction

If the training split is around 100k ECGs, `TRAIN_FRACTION = 0.01` is roughly a 1,000-ECG proof-of-concept.

In [ ]:
!python3 scripts/02_train_baseline_tabular.py \
    --data_dir data/raw \
    --label {LABEL} \
    --epochs 5 \
    --batch_size 64 \
    --seed {SEED} \
    --train_fraction {TRAIN_FRACTION}


In [ ]:
!python3 scripts/03_train_ecg_model.py \
    --data_dir data/raw \
    --label {LABEL} \
    --input_mode single \
    --lead I \
    --epochs 5 \
    --batch_size 32 \
    --seed {SEED} \
    --train_fraction {TRAIN_FRACTION}


## 9. Optional: full 12-lead smoke test

In [ ]:
# Uncomment when you want a quick 12-lead baseline
# !python3 scripts/03_train_ecg_model.py \
#     --data_dir data/raw \
#     --label {LABEL} \
#     --input_mode full12 \
#     --epochs 5 \
#     --batch_size 16 \
#     --seed {SEED} \
#     --train_fraction {TRAIN_FRACTION}


## 10. Review outputs

In [ ]:
!find outputs -maxdepth 3 -type f | sort


In [ ]:
import json
from pathlib import Path

metrics_files = sorted(Path('outputs/models').glob('*/metrics.json'))
print('Metrics files found:', len(metrics_files))
for path in metrics_files[-3:]:
    print('\n', path)
    with path.open() as f:
        metrics = json.load(f)
    print('label =', metrics['label'])
    print('input_mode =', metrics['input_mode'])
    print('lead =', metrics['lead'])
    print('test AUROC =', metrics['test_metrics']['auroc'])
    print('test AUPRC =', metrics['test_metrics']['auprc'])


## 11. Optional next commands

After the smoke test succeeds, you can increase `TRAIN_FRACTION`, train longer, switch labels, run the lead sweep, and build aggregate tables and figures.

In [ ]:
# Lead sweep for the initial labels in config
# !python3 scripts/04_run_lead_sweep.py --data_dir data/raw --epochs 5 --batch_size 32

# Label-efficiency for SHD
# !python3 scripts/05_run_label_efficiency.py --data_dir data/raw --label shd_moderate_or_greater_flag --epochs 5

# Aggregate tables and figures
# !python3 scripts/06_make_results_tables.py --output_dir outputs
